# Phase 1.4 — Data Quality

Chunked checks against the raw Retailrocket CSVs. No raw files are modified.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
            

In [2]:
# 1. Detect missing values
for name, cols in {
    'events.csv': ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'],
    'category_tree.csv': ['categoryid', 'parentid'],
}.items():
    path = RAW_DIR / name
    missing = pd.Series(dtype='int64')
    total = 0
    for chunk in pd.read_csv(path, usecols=cols, chunksize=250_000):
        total += len(chunk)
        missing = missing.add(chunk.isna().sum(), fill_value=0)
    print(f'\n{name} — {total:,} rows')
    display(missing.astype(int).to_frame('missing_count').assign(
        missing_pct=lambda x: (x['missing_count'] / total * 100).round(4)
    ))

for name in ['item_properties_part1.csv', 'item_properties_part2.csv']:
    path = RAW_DIR / name
    cols = ['timestamp', 'itemid', 'property', 'value']
    missing = pd.Series(dtype='int64')
    total = 0
    for chunk in pd.read_csv(path, usecols=cols, chunksize=250_000):
        total += len(chunk)
        missing = missing.add(chunk.isna().sum(), fill_value=0)
    print(f'\n{name} — {total:,} rows')
    display(missing.astype(int).to_frame('missing_count').assign(
        missing_pct=lambda x: (x['missing_count'] / total * 100).round(4)
    ))


events.csv — 2,756,101 rows


,missing_count,missing_pct
event,0,0.0000
itemid,0,0.0000
timestamp,0,0.0000
transactionid,2733644,99.1852
visitorid,0,0.0000



category_tree.csv — 1,669 rows


,missing_count,missing_pct
categoryid,0,0.0000
parentid,25,1.4979



item_properties_part1.csv — 10,999,999 rows


,missing_count,missing_pct
itemid,0,0.0
property,0,0.0
timestamp,0,0.0
value,0,0.0



item_properties_part2.csv — 9,275,903 rows


,missing_count,missing_pct
itemid,0,0.0
property,0,0.0
timestamp,0,0.0
value,0,0.0


In [3]:
# 2. Detect duplicate interactions
seen = set()
duplicate_rows = 0
            
for chunk in pd.read_csv(
    RAW_DIR / 'events.csv',
    usecols=['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'],
    chunksize=250_000
):
    keys = chunk.astype(object).where(chunk.notna(), None).itertuples(index=False, name=None)
    for key in keys:
        if key in seen:
            duplicate_rows += 1
        else:
            seen.add(key)

print('Exact duplicate event rows:', f'{duplicate_rows:,}')

Exact duplicate event rows: 460


In [4]:
# 3. Detect invalid product/user IDs
invalid = {'null_visitor': 0, 'null_item': 0, 'nonpositive_visitor': 0, 'nonpositive_item': 0}

for chunk in pd.read_csv(
    RAW_DIR / 'events.csv',
    usecols=['visitorid', 'itemid'],
    chunksize=250_000
):
    invalid['null_visitor'] += int(chunk['visitorid'].isna().sum())
    invalid['null_item'] += int(chunk['itemid'].isna().sum())
    invalid['nonpositive_visitor'] += int((pd.to_numeric(chunk['visitorid'], errors='coerce') <= 0).fillna(False).sum())
    invalid['nonpositive_item'] += int((pd.to_numeric(chunk['itemid'], errors='coerce') <= 0).fillna(False).sum())
            

print(pd.Series(invalid).to_frame('count'))

                     count
null_visitor             0
null_item                0
nonpositive_visitor      3
nonpositive_item         0


In [5]:
# 4. Detect inconsistent metadata
# Check whether the same item/property receives conflicting values at the same timestamp.
conflicts = 0
for name in ['item_properties_part1.csv', 'item_properties_part2.csv']:
    chunk_iter = pd.read_csv(
        RAW_DIR / name,
        usecols=['timestamp', 'itemid', 'property', 'value'],
        chunksize=250_000
    )
    for chunk in chunk_iter:
        dup = chunk.groupby(['timestamp', 'itemid', 'property'])['value'].nunique(dropna=False)
        conflicts += int((dup > 1).sum())

print('Same timestamp/item/property groups with conflicting values:', f'{conflicts:,}')

Same timestamp/item/property groups with conflicting values: 0


In [6]:
# 5. Analyze anomalous ratings/interactions
events = pd.Series(dtype='int64')
invalid_timestamps = 0
negative_timestamps = 0

for chunk in pd.read_csv(
    RAW_DIR / 'events.csv',
    usecols=['timestamp', 'event'],
    chunksize=250_000
):
    events = events.add(chunk['event'].value_counts(), fill_value=0)
    ts = pd.to_numeric(chunk['timestamp'], errors='coerce')
    invalid_timestamps += int(ts.isna().sum())
    negative_timestamps += int((ts < 0).sum())

print('Event types:')
print(events.astype(int))
print('\nInvalid timestamps:', f'{invalid_timestamps:,}')
print('Negative timestamps:', f'{negative_timestamps:,}')
print('\nNo rating field exists in the verified raw event schema; anomaly analysis therefore focuses on behavioral events and timestamps.')

Event types:
event
addtocart        69332
transaction      22457
view           2664312
dtype: int64

Invalid timestamps: 0
Negative timestamps: 0

No rating field exists in the verified raw event schema; anomaly analysis therefore focuses on behavioral events and timestamps.


## 6. Data-cleaning rules

Based on the quality checks above, final cleaning rules will be applied later during Phase 2. Raw files remain immutable.

- Preserve raw source files unchanged.
- Handle missing identifiers according to recommendation-training eligibility.
- Remove or resolve exact duplicate interaction records when justified.
- Reject invalid/non-positive identifiers from training data.
- Resolve conflicting metadata using timestamp-aware rules.
- Use chronological ordering for temporal processing.
- Treat `view`, `addtocart`, and `transaction` as implicit behavioral signals rather than ratings.
- Keep all cleaning transformations in `data/processed/`.